# Flat Interleaved OCT Model v3 — MDN Stroke Head

**Architecture:** Single causal transformer on interleaved WORD + STROKE tokens.
Stroke deltas are predicted by a **Mixture Density Network** (20 bivariate
Gaussians, Graves 2013) and **sampled** at generation time — a deterministic
regression head regresses to the mean and collapses strokes to dots (v2 failure).

**v3 fixes over v2:**
1. **MDN head** replaces deterministic xy regression — fixes stroke collapse
2. **Start-biased windows** — 20% of training windows now begin at page start
   (v2 saw a page start ~0.01% of the time, but generation always starts there)
3. **Pen weight 4→2** — v2 over-predicted pen-up, shredding strokes into dashes
4. xy_weight 10→1 (NLL scale), seq_len 1024

**Dataset:** ~1,191 OCT videos, ~12.5M tokens, ~1,237 pages.

**Before running:**
1. Runtime → Change runtime type → GPU (T4)
2. Upload `output.zip` to `My Drive/chalk-talk/a4-train/`
3. Run all cells top to bottom

Checkpoints saved as `mdn.best.pt` (the v2 `flat.best.pt` run is preserved).

In [ ]:
# ── 1. Mount Drive + Check GPU ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/chalk-talk/a4-train'
DATA_DIR   = f'{DRIVE_ROOT}/output'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'

import os, zipfile
os.makedirs(CKPT_DIR, exist_ok=True)

# Unzip if needed.
# IMPORTANT: output.zip contains files as output/*.training.jsonl,
# so we extract to DRIVE_ROOT (not DATA_DIR) to avoid nesting output/output/.
zip_path = f'{DRIVE_ROOT}/output.zip'
if os.path.exists(zip_path) and not os.path.isdir(DATA_DIR):
    print('Unzipping output.zip ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DRIVE_ROOT)    # creates DRIVE_ROOT/output/*.jsonl
    print('Done.')
elif os.path.exists(zip_path):
    # DATA_DIR exists — check if files are actually there or nested
    direct = [f for f in os.listdir(DATA_DIR) if f.endswith('.training.jsonl')]
    nested_dir = os.path.join(DATA_DIR, 'output')
    if len(direct) < 10 and os.path.isdir(nested_dir):
        nested = [f for f in os.listdir(nested_dir) if f.endswith('.training.jsonl')]
        if len(nested) > 10:
            print(f'Found {len(nested)} files in nested output/output/, moving up...')
            import shutil
            for f in nested:
                shutil.move(os.path.join(nested_dir, f), os.path.join(DATA_DIR, f))
            os.rmdir(nested_dir)
            print('Done.')

files = [f for f in os.listdir(DATA_DIR) if f.endswith('.training.jsonl')]
print(f'Found {len(files)} training files in {DATA_DIR}')
assert len(files) > 10, (
    f'ERROR: Only {len(files)} training files found. '
    f'Check zip structure — files may be nested in output/output/.'
)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)

In [ ]:
# ── 2. Model: FlatOCTModel (v3 — MDN stroke head) ────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

WORD_TYPE   = 0
STROKE_TYPE = 1
PEN_MID     = 0
PEN_UP      = 1

# Delta scaling: raw deltas have overall std ~0.07. Scale by 20 → std ~1.25,
# which is ideal for MDN: initial sigma=exp(0)=1 starts near the data scale.
DELTA_SCALE = 20.0

# MDN mixture components. Handwriting deltas are multimodal (the pen can go
# any direction next, depending on the letter) — a deterministic regression
# head predicts the MEAN of those options (~0) and strokes collapse to dots.
# A mixture density head learns the actual distribution and we SAMPLE from it.
# K=20 follows Graves 2013 handwriting synthesis.
MDN_K = 20


class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=2048, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


def causal_mask(sz, device):
    return torch.triu(torch.ones(sz, sz, device=device, dtype=torch.bool), diagonal=1)


# ── MDN helpers (bivariate Gaussian mixture, Graves 2013 eq. 23-25) ──────────

def mdn_split(params):
    """Split raw MDN params (..., 6K) into mixture components.
    Returns log_pi (..., K), mu (..., K, 2), sigma (..., K, 2), rho (..., K).
    Always computed in float32 — fp16 logsumexp/exp under AMP can NaN."""
    params = params.float()
    K = MDN_K
    log_pi  = F.log_softmax(params[..., :K], dim=-1)
    mu      = params[..., K:3*K].reshape(*params.shape[:-1], K, 2)
    log_sig = params[..., 3*K:5*K].reshape(*params.shape[:-1], K, 2).clamp(-4.0, 3.0)
    rho     = 0.95 * torch.tanh(params[..., 5*K:6*K])
    return log_pi, mu, log_sig.exp(), rho


def mdn_nll(params, target):
    """Negative log-likelihood of target (N, 2) under the GMM (N, 6K)."""
    log_pi, mu, sigma, rho = mdn_split(params)           # (N,K) (N,K,2) (N,K,2) (N,K)
    t  = target.float().unsqueeze(-2)                    # (N, 1, 2)
    zx = (t[..., 0] - mu[..., 0]) / sigma[..., 0]        # (N, K)
    zy = (t[..., 1] - mu[..., 1]) / sigma[..., 1]
    one_m_rho2 = (1 - rho ** 2).clamp(min=1e-6)
    z = zx**2 + zy**2 - 2 * rho * zx * zy
    log_gauss = (-z / (2 * one_m_rho2)
                 - torch.log(sigma[..., 0]) - torch.log(sigma[..., 1])
                 - 0.5 * torch.log(one_m_rho2) - math.log(2 * math.pi))
    return -torch.logsumexp(log_pi + log_gauss, dim=-1)  # (N,)


def mdn_sample(params, pi_temp=1.0, sigma_temp=1.0):
    """Sample one (dx, dy) from the mixture. params: flat (6K,) tensor.
    sigma_temp < 1 reduces noise (cleaner strokes), pi_temp < 1 sharpens
    component choice."""
    log_pi, mu, sigma, rho = mdn_split(params.unsqueeze(0))
    log_pi, mu, sigma, rho = log_pi[0], mu[0], sigma[0], rho[0]
    k  = int(torch.multinomial(F.softmax(log_pi / max(pi_temp, 1e-6), dim=-1), 1))
    sx = sigma[k, 0] * sigma_temp
    sy = sigma[k, 1] * sigma_temp
    r  = rho[k]
    z1, z2 = torch.randn(2, device=params.device)
    dx = mu[k, 0] + sx * z1
    dy = mu[k, 1] + sy * (r * z1 + torch.sqrt((1 - r**2).clamp(min=1e-6)) * z2)
    return float(dx), float(dy)


class FlatOCTModel(nn.Module):
    """Flat interleaved OCT model (v3). MDN stroke head, scaled delta-xy."""

    def __init__(self, vocab_size, d_model=384, n_layers=6, n_heads=8,
                 d_ff=1536, max_seq_len=2048, dropout=0.05, pad_idx=0):
        super().__init__()
        self.d_model    = d_model
        self.vocab_size = vocab_size
        self.pad_idx    = pad_idx

        self.word_embed  = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.type_embed  = nn.Embedding(2, d_model)
        self.stroke_proj = nn.Linear(2, d_model)       # scaled (dx, dy) -> d_model
        self.pen_embed   = nn.Embedding(2, d_model)
        self.pos_enc     = SinusoidalPE(d_model, max_len=max_seq_len, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm_out    = nn.LayerNorm(d_model)

        self.type_head = nn.Linear(d_model, 2)
        self.word_head = nn.Linear(d_model, vocab_size, bias=False)
        self.word_head.weight = self.word_embed.weight
        self.xy_head   = nn.Linear(d_model, 6 * MDN_K)  # MDN: pi, mu_xy, sig_xy, rho
        self.pen_head  = nn.Linear(d_model, 2)

        # Pen class weights: PEN_UP is 17.5% of stroke points. Mild 2x weight —
        # v2 used 4x which over-predicted UP at sampling time, shredding
        # strokes into the tiny dashes we saw in generation.
        self.register_buffer('pen_weight', torch.tensor([1.0, 2.0]))

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.word_embed.weight, std=0.02)
        nn.init.normal_(self.type_embed.weight, std=0.02)
        nn.init.normal_(self.pen_embed.weight, std=0.02)
        for name, p in self.named_parameters():
            if any(s in name for s in ('word_embed', 'type_embed', 'pen_embed')):
                continue
            if p.dim() > 1 and 'weight' in name:
                nn.init.xavier_uniform_(p)
            elif 'bias' in name:
                nn.init.zeros_(p)
        # Near-zero init for MDN head → log_sigma starts at 0, i.e. sigma=1,
        # right at the scaled data std (~1.25). Stable from step one.
        nn.init.normal_(self.xy_head.weight, std=0.001)
        nn.init.zeros_(self.xy_head.bias)

    def _embed(self, token_types, word_ids, xy, pen):
        h = self.type_embed(token_types)
        word_mask   = (token_types == WORD_TYPE).unsqueeze(-1).float()
        stroke_mask = (token_types == STROKE_TYPE).unsqueeze(-1).float()
        h = h + self.word_embed(word_ids) * word_mask
        h = h + (self.stroke_proj(xy) + self.pen_embed(pen)) * stroke_mask
        return self.pos_enc(h)

    def forward(self, token_types, word_ids, xy, pen, pad_mask):
        B, T = token_types.shape
        h = self._embed(token_types, word_ids, xy, pen)
        cmask  = causal_mask(T, h.device)
        hidden = self.transformer(h, mask=cmask, src_key_padding_mask=pad_mask,
                                  is_causal=True)
        hidden = self.norm_out(hidden)
        return {
            'type_logits': self.type_head(hidden),
            'word_logits': self.word_head(hidden),
            'xy_params':   self.xy_head(hidden),     # MDN params (B, T, 6K)
            'pen_logits':  self.pen_head(hidden),
        }

    def compute_loss(self, outputs, token_types, word_ids, xy, pen,
                     pad_mask, xy_weight=1.0):
        pred_type = outputs['type_logits'][:, :-1]
        pred_word = outputs['word_logits'][:, :-1]
        pred_xy   = outputs['xy_params'][:, :-1]
        pred_pen  = outputs['pen_logits'][:, :-1]

        tgt_type = token_types[:, 1:]
        tgt_word = word_ids[:, 1:]
        tgt_xy   = xy[:, 1:]
        tgt_pen  = pen[:, 1:]
        tgt_pad  = pad_mask[:, 1:]
        valid    = ~tgt_pad

        # Type loss
        l_type = F.cross_entropy(pred_type[valid], tgt_type[valid]) \
                 if valid.any() else pred_type.new_tensor(0.0)

        # Word loss with label smoothing
        word_pos = valid & (tgt_type == WORD_TYPE)
        l_word = F.cross_entropy(pred_word[word_pos], tgt_word[word_pos],
                                 ignore_index=self.pad_idx,
                                 label_smoothing=0.1) \
                 if word_pos.any() else pred_word.new_tensor(0.0)

        # XY loss: MDN negative log-likelihood (replaces Huber/MSE).
        # NOTE: NLL is a density, so it CAN GO NEGATIVE as the model
        # sharpens. Starts ~3.0, good models reach < 0. Watch it fall —
        # if it plateaus above 2.0 the stroke head isn't learning.
        stroke_pos = valid & (tgt_type == STROKE_TYPE)
        l_xy = mdn_nll(pred_xy[stroke_pos], tgt_xy[stroke_pos]).mean() \
               if stroke_pos.any() else pred_xy.new_tensor(0.0)

        # Pen loss with mild class weighting
        l_pen = F.cross_entropy(pred_pen[stroke_pos], tgt_pen[stroke_pos],
                                weight=self.pen_weight) \
                if stroke_pos.any() else pred_pen.new_tensor(0.0)

        total = l_type + l_word + xy_weight * l_xy + l_pen
        return {'total': total, 'type': l_type, 'word': l_word,
                'xy': l_xy, 'pen': l_pen}

    @torch.no_grad()
    def generate(self, seed_types, seed_words, seed_xy, seed_pen,
                 vocab, max_len=1000, temperature=0.8, top_k=50,
                 stroke_bias=0.0, pen_temperature=1.0,
                 pi_temp=1.0, sigma_temp=0.65):
        """Autoregressive generation. Strokes are SAMPLED from the MDN —
        not point estimates. sigma_temp=0.65 (Graves-style) gives cleaner
        strokes; raise toward 1.0 for more variety.
        Output contains ABSOLUTE coordinates."""
        self.eval()
        device  = seed_types.device
        id2word = {v: k for k, v in vocab.items()}
        forbidden = {vocab.get('<pad>',-1), vocab.get('<unk>',-1),
                     vocab.get('<silent>',-1), vocab.get('<page_break>',-1)}
        forbidden.discard(-1)

        types = seed_types.clone()
        words = seed_words.clone()
        xys   = seed_xy.clone()       # scaled deltas
        pens  = seed_pen.clone()
        generated = []

        # Track absolute position (accumulate unscaled seed deltas)
        abs_x, abs_y = 0.0, 0.0
        for i in range(seed_types.size(1)):
            if seed_types[0, i] == STROKE_TYPE:
                abs_x += float(seed_xy[0, i, 0]) / DELTA_SCALE
                abs_y += float(seed_xy[0, i, 1]) / DELTA_SCALE

        for _ in range(max_len):
            T = types.size(1)
            if T > 1024:
                types = types[:, -1024:]
                words = words[:, -1024:]
                xys   = xys[:, -1024:]
                pens  = pens[:, -1024:]
                T = 1024

            pad = torch.zeros(1, T, dtype=torch.bool, device=device)
            out = self.forward(types, words, xys, pens, pad)

            type_logits = out['type_logits'][0, -1].clone()
            type_logits[STROKE_TYPE] += stroke_bias
            type_probs = F.softmax(type_logits, dim=-1)
            next_type  = int(torch.multinomial(type_probs, 1))

            if next_type == WORD_TYPE:
                wl = out['word_logits'][0, -1].clone()
                for fid in forbidden:
                    wl[fid] = -float('inf')
                wl = wl / max(temperature, 1e-6)
                if top_k > 0:
                    tk_vals, tk_idx = torch.topk(wl, min(top_k, wl.size(0)))
                    mask = torch.full_like(wl, -float('inf'))
                    mask.scatter_(0, tk_idx, tk_vals)
                    wl = mask
                probs   = F.softmax(wl, dim=-1)
                word_id = int(torch.multinomial(probs, 1))
                if id2word.get(word_id) == '<eos>':
                    break
                generated.append({'type': 'word', 'word_id': word_id,
                                  'word': id2word.get(word_id, '?')})
                types = torch.cat([types, torch.tensor([[WORD_TYPE]], device=device)], 1)
                words = torch.cat([words, torch.tensor([[word_id]], device=device)], 1)
                xys   = torch.cat([xys, torch.zeros(1,1,2, device=device)], 1)
                pens  = torch.cat([pens, torch.zeros(1,1, dtype=torch.long, device=device)], 1)
            else:
                # SAMPLE a scaled delta from the mixture
                sdx, sdy = mdn_sample(out['xy_params'][0, -1],
                                      pi_temp=pi_temp, sigma_temp=sigma_temp)
                dx, dy = sdx / DELTA_SCALE, sdy / DELTA_SCALE
                new_x = max(0.0, min(1.0, abs_x + dx))
                new_y = max(0.0, min(1.0, abs_y + dy))
                actual_sdx = (new_x - abs_x) * DELTA_SCALE
                actual_sdy = (new_y - abs_y) * DELTA_SCALE
                abs_x, abs_y = new_x, new_y

                pen_logits = out['pen_logits'][0, -1].clone()
                pen_logits = pen_logits / max(pen_temperature, 1e-6)
                pen_probs  = F.softmax(pen_logits, dim=-1)
                pen_state  = int(torch.multinomial(pen_probs, 1))

                generated.append({'type': 'stroke', 'x': new_x, 'y': new_y,
                                  'pen': pen_state})
                types = torch.cat([types, torch.tensor([[STROKE_TYPE]], device=device)], 1)
                words = torch.cat([words, torch.zeros(1,1, dtype=torch.long, device=device)], 1)
                xys   = torch.cat([xys, torch.tensor([[[actual_sdx, actual_sdy]]], device=device)], 1)
                pens  = torch.cat([pens, torch.tensor([[pen_state]], device=device)], 1)

        return generated

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())

print(f'Model code loaded (v3 — MDN K={MDN_K}, delta-xy scale={DELTA_SCALE}, d=384, 6L).')

In [ ]:
# ── 3. Data: FlatOCTDataset (v3 — start-biased windows, tensor storage) ──────
import json, random, sys
from collections import Counter
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from functools import partial

PAD, UNK, BOS, EOS = '<pad>', '<unk>', '<bos>', '<eos>'
SILENT, PGBREAK    = '<silent>', '<page_break>'
SPECIALS = [PAD, UNK, BOS, EOS, SILENT, PGBREAK]

MAX_STROKES_PER_WORD = 8
MAX_POINTS_PER_WORD  = 150


def build_vocab(data_dir, min_freq=2):
    counter = Counter()
    for path in sorted(Path(data_dir).glob('*.training.jsonl')):
        for line in path.read_text().splitlines():
            if line.strip():
                tok = json.loads(line)
                if tok['type'] == 'word':
                    counter[tok['word']] += 1
    vocab = {s: i for i, s in enumerate(SPECIALS)}
    for word, freq in counter.most_common():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab


def augment_delta_xy(dxy, scale_lo=0.85, scale_hi=1.15, jitter=0.1):
    """Scale + jitter SCALED delta coordinates. No clamping."""
    if dxy.size(0) == 0: return dxy
    scale = random.uniform(scale_lo, scale_hi)
    return dxy * scale + torch.randn_like(dxy) * jitter


def abs_to_delta_scaled(types, xys, scale=DELTA_SCALE):
    """Convert absolute xy to SCALED delta-xy for stroke tokens."""
    delta_xys = []
    last_x, last_y = 0.0, 0.0
    for t, xy in zip(types, xys):
        if t == STROKE_TYPE:
            dx = (xy[0] - last_x) * scale
            dy = (xy[1] - last_y) * scale
            delta_xys.append([dx, dy])
            last_x, last_y = xy[0], xy[1]
        else:
            delta_xys.append([0.0, 0.0])
    return delta_xys


def flatten_page(tokens, vocab,
                 max_points=MAX_POINTS_PER_WORD,
                 max_strokes=MAX_STROKES_PER_WORD):
    """Convert word-anchored tokens into flat interleaved sequence.
    Returns tensors (not lists) for memory efficiency."""
    unk_id    = vocab[UNK]
    silent_id = vocab[SILENT]
    types, wids, abs_xys, pens = [], [], [], []

    for tok in tokens:
        t = tok['type']
        if t == 'word':
            wid = vocab.get(tok['word'], unk_id)
            types.append(WORD_TYPE); wids.append(wid)
            abs_xys.append([0.0, 0.0]); pens.append(0)
        elif t == 'silent':
            types.append(WORD_TYPE); wids.append(silent_id)
            abs_xys.append([0.0, 0.0]); pens.append(0)
        else:
            continue

        n_pts = 0
        for stroke in tok.get('strokes', [])[:max_strokes]:
            for i, pt in enumerate(stroke):
                if n_pts >= max_points: break
                x, y = float(pt[0]), float(pt[1])
                pen = PEN_UP if (i == len(stroke) - 1) else PEN_MID
                types.append(STROKE_TYPE); wids.append(0)
                abs_xys.append([x, y]); pens.append(pen)
                n_pts += 1
            if n_pts >= max_points: break

    scaled_delta_xys = abs_to_delta_scaled(types, abs_xys)

    # Store as tensors — 4-6x less RAM than Python lists for 12M+ tokens
    return {
        'token_types': torch.tensor(types, dtype=torch.long),
        'word_ids':    torch.tensor(wids, dtype=torch.long),
        'xys':         torch.tensor(scaled_delta_xys, dtype=torch.float32),
        'pens':        torch.tensor(pens, dtype=torch.long),
    }


class FlatOCTDataset(Dataset):
    """One item = one page. Returns a window of seq_len tokens.

    WINDOW SAMPLING FIX (v3): pages average ~9,300 tokens, so a uniformly
    random 1,022-token window starts at position 0 with probability ~0.01%.
    The v2 model NEVER saw page starts during training — yet generation
    always seeds from [BOS + intro words]. That train/inference mismatch
    is a big reason generation collapsed into word spam.
    Now: 20% of windows start at position 0 (page start, with BOS),
    10% end at the last token (with EOS), 70% uniform random.
    """

    def __init__(self, data_dir, vocab, seq_len=512, augment=False,
                 unk_rate=0.05, min_tokens=20, start_frac=0.2, end_frac=0.1):
        self.vocab      = vocab
        self.seq_len    = seq_len
        self.augment    = augment
        self.unk_rate   = unk_rate
        self.start_frac = start_frac
        self.end_frac   = end_frac
        self.pad_id    = vocab[PAD];     self.unk_id  = vocab[UNK]
        self.bos_id    = vocab[BOS];     self.eos_id  = vocab[EOS]
        self.silent_id = vocab[SILENT];  self.pgbrk_id = vocab[PGBREAK]
        self.pages = []
        self._load(data_dir, min_tokens)

    def _load(self, data_dir, min_tokens):
        paths = sorted(Path(data_dir).glob('*.training.jsonl'))
        n_files = len(paths)
        for fi, path in enumerate(paths):
            if fi % 100 == 0:
                print(f'\r  Loading file {fi+1}/{n_files}...', end='', flush=True)
            lines = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
            current, meta = [], {}
            for tok in lines:
                if tok['type'] == 'lesson_start':
                    current = []; meta = tok
                elif tok['type'] in ('page_break', 'end'):
                    if current:
                        flat = flatten_page(current, self.vocab)
                        if flat['token_types'].size(0) >= min_tokens:
                            flat['topic'] = meta.get('topic', '')
                            flat['tag']   = meta.get('tag', '')
                            self.pages.append(flat)
                    current = []
                else:
                    current.append(tok)
            if current:
                flat = flatten_page(current, self.vocab)
                if flat['token_types'].size(0) >= min_tokens:
                    flat['topic'] = meta.get('topic', '')
                    flat['tag']   = meta.get('tag', '')
                    self.pages.append(flat)

        total_tok = sum(p['token_types'].size(0) for p in self.pages)
        print(f'\r[dataset] {len(self.pages)} pages from {n_files} videos '
              f'({total_tok:,} total tokens, augment={self.augment})')

    def __len__(self): return len(self.pages)

    def __getitem__(self, idx):
        page = self.pages[idx]
        total_len = page['token_types'].size(0)
        effective = self.seq_len - 2

        if total_len <= effective:
            start, end = 0, total_len
            add_bos = add_eos = True
        else:
            r = random.random()
            if r < self.start_frac:
                # Page start — the context generation always begins from
                start = 0
            elif r < self.start_frac + self.end_frac:
                # Page end — teaches the model where EOS happens
                start = total_len - effective
            else:
                start = random.randint(0, total_len - effective)
            end   = start + effective
            add_bos = (start == 0)
            add_eos = (end == total_len)

        # Slice tensors directly (fast, no Python list overhead)
        tt = page['token_types'][start:end]
        wi = page['word_ids'][start:end]
        xy = page['xys'][start:end]
        pn = page['pens'][start:end]

        # Prepend BOS / append EOS via cat
        if add_bos:
            tt = torch.cat([torch.tensor([WORD_TYPE]), tt])
            wi = torch.cat([torch.tensor([self.bos_id]), wi])
            xy = torch.cat([torch.zeros(1, 2), xy])
            pn = torch.cat([torch.tensor([0]), pn])
        if add_eos:
            tt = torch.cat([tt, torch.tensor([WORD_TYPE])])
            wi = torch.cat([wi, torch.tensor([self.eos_id])])
            xy = torch.cat([xy, torch.zeros(1, 2)])
            pn = torch.cat([pn, torch.tensor([0])])

        # Clone for augmentation (avoid modifying stored data)
        token_types = tt.clone()
        word_ids    = wi.clone()
        xy_out      = xy.clone()
        pen         = pn.clone()

        if self.augment:
            specials = {self.pad_id, self.bos_id, self.eos_id,
                        self.silent_id, self.pgbrk_id}
            for i in range(len(word_ids)):
                if token_types[i] == WORD_TYPE and word_ids[i].item() not in specials:
                    if random.random() < self.unk_rate:
                        word_ids[i] = self.unk_id
            sm = (token_types == STROKE_TYPE)
            if sm.any():
                xy_out[sm] = augment_delta_xy(xy_out[sm])

        return {'token_types': token_types, 'word_ids': word_ids,
                'xy': xy_out, 'pen': pen}


def flat_collate_fn(batch, pad_id=0):
    B     = len(batch)
    T_max = max(b['token_types'].size(0) for b in batch)
    token_types = torch.zeros(B, T_max, dtype=torch.long)
    word_ids    = torch.full((B, T_max), pad_id, dtype=torch.long)
    xy          = torch.zeros(B, T_max, 2)
    pen         = torch.zeros(B, T_max, dtype=torch.long)
    pad_mask    = torch.ones(B, T_max, dtype=torch.bool)
    for i, b in enumerate(batch):
        L = b['token_types'].size(0)
        token_types[i, :L] = b['token_types']
        word_ids[i, :L]    = b['word_ids']
        xy[i, :L]          = b['xy']
        pen[i, :L]         = b['pen']
        pad_mask[i, :L]    = False
    return {'token_types': token_types, 'word_ids': word_ids,
            'xy': xy, 'pen': pen, 'pad_mask': pad_mask}

print(f'Data code loaded (v3 — start-biased windows, delta-xy scale={DELTA_SCALE}).')

In [ ]:
# ── 4. Config ─────────────────────────────────────────────────────────────────
# v3 changes:
#   xy_weight = 1.0  — MDN NLL is already on a comparable scale to CE losses
#                      (starts ~3, falls below 0). The old 10.0 was tuned for
#                      tiny Huber values and would swamp everything.
#   Loss totals are NOT comparable to v2 runs (different xy loss + weight).
#   Watch val 'xy' — it should fall steadily; below 0 is good.
CFG = dict(
    d_model    = 384,
    n_layers   = 6,
    n_heads    = 8,
    d_ff       = 1536,
    dropout    = 0.05,
    seq_len    = 1024,    # 92% of pages are longer than this — full windows
    batch_size = 8,
    lr         = 3e-4,
    xy_weight  = 1.0,     # MDN NLL scale (was 10.0 for Huber)
    epochs     = 40,
    warmup     = 300,     # ~2 epochs
    patience   = 12,
    val_frac   = 0.1,
    min_freq   = 2,       # 11k vocab, 99.7% coverage
    grad_clip  = 1.0,
)
print('Config:', CFG)

In [ ]:
# ── 5. Build vocab + dataset ──────────────────────────────────────────────────
import glob

vocab_path = f'{CKPT_DIR}/vocab.json'
n_training_files = len([f for f in os.listdir(DATA_DIR) if f.endswith('.training.jsonl')])

# Delete stale vocab/checkpoints if vocab was built on different data
rebuild_vocab = False
if os.path.exists(vocab_path):
    old = json.loads(open(vocab_path).read())
    if len(old) < 50:
        print(f'Stale vocab ({len(old)} tokens), rebuilding...')
        rebuild_vocab = True
    else:
        print(f'Loaded existing vocab: {len(old)} tokens')
        vocab = old
else:
    rebuild_vocab = True

if rebuild_vocab:
    for f in glob.glob(f'{CKPT_DIR}/*.pt'):
        print(f'  Deleting stale checkpoint: {os.path.basename(f)}')
        os.remove(f)
    for f in glob.glob(f'{CKPT_DIR}/*.log.jsonl'):
        os.remove(f)
    if os.path.exists(vocab_path):
        os.remove(vocab_path)

    print(f'Building vocab from {n_training_files} files (min_freq={CFG["min_freq"]})...')
    vocab = build_vocab(DATA_DIR, min_freq=CFG['min_freq'])
    assert len(vocab) > 100, f'Vocab has only {len(vocab)} tokens — check DATA_DIR'
    open(vocab_path, 'w').write(json.dumps(vocab, ensure_ascii=False))
    print(f'Built vocab: {len(vocab)} tokens')

pad_id = vocab[PAD]
print(f'Vocab size: {len(vocab)}  (from {n_training_files} training files)')

# Build dataset
print(f'\nLoading dataset from {n_training_files} files...')
full_ds = FlatOCTDataset(DATA_DIR, vocab, seq_len=CFG['seq_len'], augment=False)

assert len(full_ds) > 50, (
    f'Only {len(full_ds)} pages loaded — expected ~1,000+. '
    f'Check if DATA_DIR points to the right location.'
)

n_val   = max(1, int(len(full_ds) * CFG['val_frac']))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_ds.dataset.augment = True

coll     = partial(flat_collate_fn, pad_id=pad_id)
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                      collate_fn=coll, num_workers=2, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                      collate_fn=coll, num_workers=2)

print(f'Train: {n_train}  Val: {n_val}  Batches/epoch: {len(train_dl)}')

# ── Data sanity check ────────────────────────────────────────────────────────
import numpy as np
all_pen_mid, all_pen_up = 0, 0
all_sdx, all_sdy = [], []
page_lens = []

for i in range(min(50, len(full_ds))):
    s = full_ds[i]
    page_lens.append(s['token_types'].size(0))
    sm = (s['token_types'] == STROKE_TYPE)
    if sm.any():
        sxy = s['xy'][sm]
        all_sdx.extend(sxy[:, 0].tolist())
        all_sdy.extend(sxy[:, 1].tolist())
        pv = s['pen'][sm]
        all_pen_mid += (pv == PEN_MID).sum().item()
        all_pen_up  += (pv == PEN_UP).sum().item()

print(f'\nSanity check (first 50 pages):')
print(f'  Page lengths: min={min(page_lens)} max={max(page_lens)} mean={sum(page_lens)/len(page_lens):.0f}')
sdx_arr = np.array(all_sdx)
sdy_arr = np.array(all_sdy)
print(f'  Scaled dx: mean={sdx_arr.mean():.3f} std={sdx_arr.std():.3f} range=[{sdx_arr.min():.1f}, {sdx_arr.max():.1f}]')
print(f'  Scaled dy: mean={sdy_arr.mean():.3f} std={sdy_arr.std():.3f} range=[{sdy_arr.min():.1f}, {sdy_arr.max():.1f}]')
print(f'  Pen: MID={all_pen_mid} ({100*all_pen_mid/(all_pen_mid+all_pen_up):.1f}%) UP={all_pen_up} ({100*all_pen_up/(all_pen_mid+all_pen_up):.1f}%)')

# MDN NLL baseline: a single Gaussian with the data's own std.
# The MDN should beat this within a few epochs (more components = sharper fit).
nll_baseline = (math.log(2*math.pi) + math.log(max(sdx_arr.std(), 1e-6))
                + math.log(max(sdy_arr.std(), 1e-6)) + 1.0)
print(f'  MDN NLL baseline (single Gaussian) ≈ {nll_baseline:.2f} — '
      f'val xy should drop well below this')

In [ ]:
# ── 6. Build model ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

model = FlatOCTModel(
    vocab_size=len(vocab),
    d_model=CFG['d_model'],
    n_layers=CFG['n_layers'],
    n_heads=CFG['n_heads'],
    d_ff=CFG['d_ff'],
    max_seq_len=max(CFG['seq_len'] * 2, 2048),
    dropout=CFG['dropout'],
    pad_idx=pad_id,
).to(device)

print(f'Model: {model.n_params/1e6:.2f}M params')
if device == 'cuda':
    print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── 7. Training ───────────────────────────────────────────────────────────────
import time, math, glob
from tqdm.notebook import tqdm

# v3 checkpoints use the mdn.* prefix — the v2 run (flat.*) is left intact
# for comparison.
log_path   = f'{CKPT_DIR}/mdn.log.jsonl'
ckpt_path  = f'{CKPT_DIR}/mdn.best.pt'
for f in glob.glob(f'{CKPT_DIR}/mdn.*'):
    os.remove(f)
    print(f'Deleted {os.path.basename(f)}')

XY_W = CFG['xy_weight']


def step_batch(model, batch):
    tt  = batch['token_types'].to(device)
    wi  = batch['word_ids'].to(device)
    xy  = batch['xy'].to(device)
    pen = batch['pen'].to(device)
    pm  = batch['pad_mask'].to(device)
    out = model(tt, wi, xy, pen, pm)
    losses = model.compute_loss(out, tt, wi, xy, pen, pm, xy_weight=XY_W)
    return losses['total'], {k: v.item() for k, v in losses.items()}


opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=0.01)

total_steps = CFG['epochs'] * len(train_dl)
warmup_steps = CFG['warmup']

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * progress))

sched  = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None
use_amp = (device == 'cuda')

best_val   = float('inf')
no_improve = 0
log_f      = open(log_path, 'a')

print(f'Training: {CFG["epochs"]} epochs, {len(train_dl)} batches/epoch')
print(f'Warmup: {warmup_steps} steps, Total: {total_steps} steps')

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()
    model.train()
    tr = {}; nb = 0

    for batch in tqdm(train_dl, desc=f'ep{epoch:03d}/{CFG["epochs"]}', leave=False):
        opt.zero_grad(set_to_none=True)
        if use_amp:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                total, parts = step_batch(model, batch)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(opt)
            scaler.update()
        else:
            total, parts = step_batch(model, batch)
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            opt.step()
        sched.step()
        for k, v in parts.items(): tr[k] = tr.get(k, 0.) + v
        nb += 1

    # Validation
    model.eval()
    vl = {}; nv = 0
    with torch.no_grad():
        for batch in val_dl:
            if use_amp:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    _, parts = step_batch(model, batch)
            else:
                _, parts = step_batch(model, batch)
            for k, v in parts.items(): vl[k] = vl.get(k, 0.) + v
            nv += 1

    ta = {k: v / max(nb, 1) for k, v in tr.items()}
    va = {k: v / max(nv, 1) for k, v in vl.items()}
    vl_total = va.get('total', float('inf'))
    elapsed  = time.time() - t0

    rec = {'epoch': epoch, 'train': ta, 'val': va, 'lr': sched.get_last_lr()[0],
           'elapsed': round(elapsed, 1)}
    log_f.write(json.dumps(rec) + '\n'); log_f.flush()

    print(f'  ep{epoch:03d}  '
          f'train={ta.get("total",0):.4f}  '
          f'val={vl_total:.4f}  '
          f'(type={va.get("type",0):.3f} '
          f'word={va.get("word",0):.3f} '
          f'xy={va.get("xy",0):+.4f} '
          f'pen={va.get("pen",0):.3f})  '
          f'({elapsed:.0f}s)')

    if vl_total < best_val:
        best_val = vl_total; no_improve = 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_total,
                    'config': {'vocab_size': len(vocab), 'd_model': CFG['d_model'],
                               'n_layers': CFG['n_layers'], 'n_heads': CFG['n_heads'],
                               'mdn_k': MDN_K}},
                   ckpt_path)
        print(f'    -> saved (val={vl_total:.4f})')
    else:
        no_improve += 1
        if no_improve >= CFG['patience']:
            print(f'    early stop'); break

log_f.close()
print(f'\nTraining done. Best val: {best_val:.4f}')

In [ ]:
# ── 8. Generation test ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import random as _random

# Load best checkpoint
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]} (val={ckpt["val_loss"]:.4f})')

id2word = {v: k for k, v in vocab.items()}

# Pick a random validation page for ground truth comparison
ep_idx  = _random.choice(range(len(val_ds)))
page    = full_ds.pages[val_ds.indices[ep_idx]]
print(f'Page: {page.get("tag","")} | {page.get("topic","")}')

# Build trigger from first few WORD tokens in the page
# NOTE: page data stores tensors, so use .item() to convert to int for dict lookup
trigger_words = []
for i in range(len(page['token_types'])):
    t = page['token_types'][i].item()
    w = page['word_ids'][i].item()
    if t == WORD_TYPE:
        trigger_words.append(id2word.get(w, '?'))
    if len(trigger_words) >= 5:
        break

# Encode trigger as seed
seed_types = [WORD_TYPE]  # BOS
seed_wids  = [vocab[BOS]]
seed_xys   = [[0.0, 0.0]]
seed_pens  = [0]
for w in trigger_words:
    wid = vocab.get(w, vocab[UNK])
    seed_types.append(WORD_TYPE)
    seed_wids.append(wid)
    seed_xys.append([0.0, 0.0])
    seed_pens.append(0)

print(f'Trigger: {" ".join(trigger_words)}')
print('Generating...')

generated = model.generate(
    seed_types=torch.tensor([seed_types], dtype=torch.long, device=device),
    seed_words=torch.tensor([seed_wids], dtype=torch.long, device=device),
    seed_xy=torch.tensor([seed_xys], dtype=torch.float32, device=device),
    seed_pen=torch.tensor([seed_pens], dtype=torch.long, device=device),
    vocab=vocab,
    max_len=800,
    temperature=0.8,
    top_k=40,
)

# Split generated into words and strokes
gen_words   = [g['word'] for g in generated if g['type'] == 'word']
gen_strokes = [(g['x'], g['y'], g['pen']) for g in generated if g['type'] == 'stroke']
print(f'Generated: {len(gen_words)} words, {len(gen_strokes)} stroke points')
print(f'Words: {" ".join(gen_words[:30])}...')

In [ ]:
# ── 9. Visualize: Ground Truth vs Generated ──────────────────────────────────

def delta_to_abs(types_list, dxy_list, scale=DELTA_SCALE):
    """Convert SCALED delta-xy back to absolute for visualization.
    Divides by DELTA_SCALE before accumulating."""
    abs_xys = []
    ax, ay = 0.0, 0.0
    for t, (dx, dy) in zip(types_list, dxy_list):
        # Handle tensor or int types
        t_val = t.item() if hasattr(t, 'item') else t
        dx_val = dx.item() if hasattr(dx, 'item') else float(dx)
        dy_val = dy.item() if hasattr(dy, 'item') else float(dy)
        if t_val == STROKE_TYPE:
            ax += dx_val / scale
            ay += dy_val / scale
            abs_xys.append((ax, ay))
        else:
            abs_xys.append((0.0, 0.0))
    return abs_xys

def draw_flat_strokes(ax, types_list, xys_list, pens_list, title, color='blue'):
    """Draw stroke points from flat interleaved sequence (absolute coords)."""
    ax.set_facecolor('#f5f5f5')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

    current_stroke = []
    for t, (x, y), p in zip(types_list, xys_list, pens_list):
        t_val = t.item() if hasattr(t, 'item') else t
        p_val = p.item() if hasattr(p, 'item') else p
        x_val = x.item() if hasattr(x, 'item') else float(x)
        y_val = y.item() if hasattr(y, 'item') else float(y)
        if t_val == STROKE_TYPE:
            current_stroke.append((x_val, 1.0 - y_val))
            if p_val == PEN_UP:
                if len(current_stroke) > 1:
                    xs, ys = zip(*current_stroke)
                    ax.plot(xs, ys, color=color, linewidth=1.3,
                            alpha=0.85, solid_capstyle='round')
                current_stroke = []
    if len(current_stroke) > 1:
        xs, ys = zip(*current_stroke)
        ax.plot(xs, ys, color=color, linewidth=1.3, alpha=0.85)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"{page.get('tag','')} | {page.get('topic','')[:55]}",
             fontsize=11, y=1.01)

# Ground truth — convert SCALED delta-xy back to absolute for drawing
gt_types = page['token_types']
gt_dxys  = page['xys']      # these are SCALED deltas (multiplied by DELTA_SCALE)
gt_pens  = page['pens']
gt_wids  = page['word_ids']

gt_limit = min(len(gt_types), len(generated) + len(seed_types))
gt_abs_xys = delta_to_abs(gt_types[:gt_limit], gt_dxys[:gt_limit])

draw_flat_strokes(axes[0], gt_types[:gt_limit], gt_abs_xys, gt_pens[:gt_limit],
                  f'Ground Truth (first {gt_limit} tokens)', color='#1a3a8c')

# FIX: use .item() to convert tensor word IDs to ints for dict lookup
gt_words = [id2word.get(int(w), '?')
            for t, w in zip(gt_types[:gt_limit], gt_wids[:gt_limit])
            if int(t) == WORD_TYPE and int(w) not in (vocab[BOS], vocab[EOS], vocab[PAD])]
gt_label = ' '.join(gt_words[:40])
axes[0].text(0.5, -0.04, (gt_label[:90]+'...' if len(gt_label) > 90 else gt_label),
             transform=axes[0].transAxes, fontsize=6.5, ha='center',
             va='top', color='#444')

# Generated (already has absolute coords from generate())
gen_types_list = [STROKE_TYPE if g['type'] == 'stroke' else WORD_TYPE for g in generated]
gen_xys_list   = [(g.get('x', 0), g.get('y', 0)) for g in generated]
gen_pens_list  = [g.get('pen', 0) for g in generated]
draw_flat_strokes(axes[1], gen_types_list, gen_xys_list, gen_pens_list,
                  'Generated', color='#8c1a1a')

gen_label = ' '.join(gen_words[:40])
axes[1].text(0.5, -0.04, (gen_label[:90]+'...' if len(gen_label) > 90 else gen_label),
             transform=axes[1].transAxes, fontsize=6.5, ha='center',
             va='top', color='#444')

plt.tight_layout()
save_path = f"{CKPT_DIR}/gen_v2_{page.get('tag','test')}.png"
plt.savefig(save_path, dpi=130, bbox_inches='tight')
plt.show()

# Stats
n_gt_strokes = sum(1 for t in gt_types[:gt_limit] if int(t) == STROKE_TYPE)
n_gen_strokes = len(gen_strokes)
print(f'Ground truth: {n_gt_strokes} stroke pts in {gt_limit} tokens')
print(f'Generated:    {n_gen_strokes} stroke pts in {len(generated)} tokens')
print(f'Saved to {save_path}')